# Limpieza del archivo csv con ejercicios

In [1]:
import pandas as pd
import os
from pathlib import Path
import shutil
import re
import unicodedata

## Ruta de directorio raíz

In [2]:
try:
    root = Path("/home/daniel/code/clasificador_selectividad")
    os.chdir(root)
except Exception as e:
    root = Path("/Users/daniel/code/clasificador_selectividad")
    os.chdir(root)

## Funciones

quitar acentos

In [3]:
def remove_accent_marks(text: str) -> str:
    nfd = unicodedata.normalize("NFD", text)
    return "".join(c for c in nfd if not unicodedata.combining(c))

crear id

In [ ]:
def create_id(row):
    partes = [row["asignatura"], row["tema"], row["año"], row["convocatoria"]]
    if pd.notna(row["ejercicio"]):
        partes.append(row["ejercicio"])
    # se usa str() para convertir año (int) en string
    id_string = "_".join(str(p) for p in partes).lower()

    # quitar acentos: descomponer y eliminar los caracteres combinantes
    id_string = unicodedata.normalize("NFKD", id_string)
    id_string = "".join(c for c in id_string if not unicodedata.combining(c))

    # dejar solo letras, números y _ (quita comas, puntos, espacios...)
    id_string = re.sub(r"[^a-z0-9]+", "_", id_string).strip("_")
    return id_string

## Cargar archivo csv

In [4]:
df = pd.read_csv(root / "csv/ejercicios_2026.csv", encoding="utf-8")

dataframe de ejercicios y dataframe con ejercicios no parseados

In [5]:
df_sin_metadatos = df[df.aviso == "sin metadatos"]
df = df[df.aviso.isna()]

In [6]:
df_sin_metadatos

,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen,aviso


In [7]:
df.sample(3)

,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen,aviso
132,Mates II,Mates II,2026,Reserva 3,Ejercicio 1,Funciones,MATEMÁTICAS II. 2026. RESERVA 3. EJERCICIO 1,En un jardín botánico se desea crear un recint...,source_pdf_files/Mates II/funciones/2026 - Fun...,enunciados_ejercicios/Mates II/funciones/png/p...,NaN
32,Mates II,Mates II,2026,Junio,Ejercicio 3.2,Matrices Y Determinantes,MATEMÁTICAS II. 2026. JUNIO. EJERCICIO 3.2,Considera la matriz A =\n \n2\n3\n\n\n...,source_pdf_files/Mates II/matrices_determinant...,enunciados_ejercicios/Mates II/matrices_determ...,NaN
95,Física,Fisica,2026,Junio,Ejercicio A1,Física cuántica y nuclear,FISICA. 2026. JUNIO. EJERCICIO a1,"g = 9,8 m s-2\nSe produce emisión de fotoelect...",source_pdf_files/Fisica/fisica_cuantica_nuclea...,enunciados_ejercicios/Fisica/fisica_cuantica_n...,NaN


## Inspección previa a limpieza

comprobar si coinciden los valores de `asignatura` y `asignatura_carpeta`

In [8]:
if "asignatura_carpeta" in df.columns:
    valores_asignatura = df.asignatura.apply(remove_accent_marks)
    valores_asignatura_carpeta = df.asignatura_carpeta.str.normalize("NFC")

    if all(valores_asignatura == valores_asignatura_carpeta):
        print("Valores coinciden. Elimino columna asignatura_carpeta")
        df = df.drop("asignatura_carpeta", axis = 1)

Valores coinciden. Elimino columna asignatura_carpeta


In [9]:
df.asignatura.unique()

<ArrowStringArray>
['Mates CCSS', 'Mates II', 'Química', 'Física']
Length: 4, dtype: str

In [10]:
df.convocatoria.unique()

<ArrowStringArray>
['Modelo', 'Junio', 'Reserva 1', 'Reserva 2', 'Reserva 3', 'Reserva 4',
 'Julio']
Length: 7, dtype: str

In [11]:
df[df.convocatoria.isna()]

,asignatura,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen,aviso


In [12]:
print(*df[df.convocatoria.isna()].info_ejercicio, sep = "\n")

In [13]:
print(*df[df.convocatoria.isna()].archivo, sep = "\n")

In [37]:
df.enunciado.isna().sum()

np.int64(0)

En este caso, los enunciados vacíos se deben a enunciado de ejercicio competencial en forma de imagen y no texto

In [ ]:
# df.enunciado = df.enunciado.fillna("ejercicio competencial, enunciado imagen")

## Limpieza

convertir a tipo oración los nombres de los temas

In [ ]:
df["tema"] = df["tema"].str.capitalize().str.strip()

añadir acentos a los nombres de tema

In [15]:
correcciones_nombre_tema = {
    "Distribucion binomial": "Distribución binomial",
    "Inferencia estadistica": "Inferencia estadística",
    "Reactividad organica": "Reactividad orgánica",
    "Acido base": "Ácido - base",
    "Equilibrio quimico": "Equilibrio químico",
    "Geometria": "Geometría",
    "Distribucion normal y binomial": "Distribución normal y binomial",
    "Conf. electrónica" : "Configuración electrónica",
}

df["tema"] = df.tema.replace(correcciones_nombre_tema)

In [16]:
df[["asignatura", "tema"]].drop_duplicates()

,asignatura,tema
0,Mates CCSS,Distribución binomial
7,Mates CCSS,Programación lineal
12,Mates CCSS,Sistemas de ecuaciones lineales
14,Mates CCSS,Funciones ccss
24,Mates CCSS,Inferencia estadística
32,Mates II,Matrices y determinantes
35,Mates II,Probabilidad
39,Química,Enlace químico
43,Química,Configuración electrónica
51,Química,Reactividad orgánica


In [17]:
print(df[df.ejercicio.isna()].info_ejercicio)

Series([], Name: info_ejercicio, dtype: str)


año a `integer`

In [ ]:
df["año"] = df["año"].astype(int)

pasar "Opción" a minúsculas

In [ ]:
df["ejercicio"] = df["ejercicio"].apply(lambda x : x.replace("Opción", "opción")
                                      .replace(".", ",") if pd.notna(x) else x)


eliminar datos de gravedad de enunciados de física

In [20]:
pattern = r"g\s*=\s*9[,.]8\s*m\s*[·.]?\s*s\s*-\s*2\s*"
mask = df.asignatura == "Física"
df.loc[mask, "enunciado"] = df.loc[mask, "enunciado"].str.replace(pattern, "", regex=True)

Normalizar texto

In [84]:
# df.enunciado = df.enunciado.apply(lambda texto : unicodedata.normalize("NFKC", texto)
#                                       if pd.notna(texto) else texto)

Eliminar espacios múltiples de los enunciados.

In [38]:
df["enunciado"] = df["enunciado"].apply(lambda texto : re.sub(r"[ \t]+", " ", texto)
                                      .strip() if pd.notna(texto) else texto)

Eliminar "," al final del nombre de ejercicio

In [39]:
df["ejercicio"] = df["ejercicio"].str.strip(",")

Tras la limpieza completa, creo el ID

In [28]:
df["id"] = df.apply(lambda row: create_id(row), axis = 1)

Reorganizar las columnas

In [31]:
df.columns

Index(['id', 'asignatura', 'año', 'convocatoria', 'ejercicio', 'tema',
       'info_ejercicio', 'enunciado', 'archivo', 'imagen'],
      dtype='str')

In [30]:
df = df[["id"] + list(df.columns[:-2])]

## Exportar dataframe limpio

In [32]:
df = (df.sort_values(["asignatura", "tema", "año", "convocatoria", "ejercicio"],
               ascending = [True, True, False, True, True])
               .reset_index(drop = True)
)

In [40]:
df.sample(5)

,id,asignatura,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen
125,quimica_termoquimica_2026_julio_ejercicio_2a,Química,2026,Julio,Ejercicio 2A,Termoquímica,QUIMICA. 2026. JULIO. EJERCICIO 2A,Razone si las siguientes afirmaciones son verd...,source_pdf_files/Quimica/termoquimica/2026 - T...,enunciados_ejercicios/Quimica/termoquimica/png...
101,quimica_reacciones_redox_2026_junio_ejercicio_5,Química,2026,Junio,Ejercicio 5,Reacciones redox,QUÍMICA. 2026. JUNIO. EJERCICIO 5,"ejercicio competencial, enunciado imagen",source_pdf_files/Quimica/redox/2026 - Reaccion...,enunciados_ejercicios/Quimica/redox/png/png_Qu...
98,quimica_equilibrio_quimico_2026_reserva_3_ejer...,Química,2026,Reserva 3,Ejercicio 2B,Equilibrio químico,QUÍMICA. 2026. RESERVA 3. EJERCICIO 2B,Dado el siguiente sistema en equilibrio:\n3 F ...,source_pdf_files/Quimica/equilibrio_quimico/20...,enunciados_ejercicios/Quimica/equilibrio_quimi...
42,mates_ccss_programacion_lineal_2026_reserva_3_...,Mates CCSS,2026,Reserva 3,Ejercicio 1B,Programación lineal,SOCIALES II. 2026 RESERVA 3. EJERCICIO 1B,Consideremos el recinto definido por las sigui...,source_pdf_files/Mates CCSS/programacion_linea...,enunciados_ejercicios/Mates CCSS/programacion_...
108,quimica_reacciones_redox_2026_reserva_4_ejerci...,Química,2026,Reserva 4,Ejercicio 3B,Reacciones redox,QUÍMICA. 2026. RESERVA 4. EJERCICIO 3B,a) ¿Cuánta carga eléctrica se consume al depos...,source_pdf_files/Quimica/redox/2026 - Reaccion...,enunciados_ejercicios/Quimica/redox/png/png_Qu...


In [41]:
df.to_csv(root/"csv/ejercicios_2026_clean.csv", index = False, encoding="utf-8")

## Revisión archivo limpio

In [42]:
df = pd.read_csv(root / "csv/ejercicios_2026_clean.csv", encoding="utf-8")

In [43]:
df.sample(3)

,id,asignatura,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen
129,quimica_acido_base_2026_julio_ejercicio_5,Química,2026,Julio,Ejercicio 5,Ácido - base,QUÍMICA. 2026. JULIO. EJERCICIO 5,"ejercicio competencial, enunciado imagen",source_pdf_files/Quimica/acido_base/2026 - Aci...,enunciados_ejercicios/Quimica/acido_base/png/p...
81,mates_ii_sistemas_de_ecuaciones_lineales_2026_...,Mates II,2026,Reserva 4,"Ejercicio 4,1",Sistemas de ecuaciones lineales,MATEMÁTICAS II. 2026. RESERVA 4. EJERCICIO 4.1,Considera el sistema de ecuaciones\n x\n7...,source_pdf_files/Mates II/sistemas_de_ecuacion...,enunciados_ejercicios/Mates II/sistemas_de_ecu...
4,fisica_campo_gravitatorio_2026_junio_ejercicio_a,Física,2026,Junio,Ejercicio A,Campo gravitatorio,FISICA. 2026. JUNIO. EJERCICIO a,Razone la veracidad de las siguientes afirmaci...,source_pdf_files/Fisica/campo_gravitatorio/202...,enunciados_ejercicios/Fisica/campo_gravitatori...
